# Multimodal Embedding with Qwen3-VL and OpenVINO

The [Qwen3-VL-Embedding model series](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B) is built upon the powerful Qwen3-VL foundation model, specifically designed for multimodal embedding tasks. It accepts diverse inputs including text, images, screenshots, and videos, as well as inputs containing a mixture of these modalities. The model generates high-dimensional vectors for broad applications like retrieval, clustering, and classification.

<img src="https://model-demo.oss-cn-hangzhou.aliyuncs.com/Qwen3-VL-Embedding.png" width="500"/>

In this tutorial we consider how to convert and optimize Qwen3-VL Embedding model using OpenVINO.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert model using Optimum Intel](#Convert-model-using-Optimum-Intel)
- [Run OpenVINO model inference with Optimum-intel](#Run-OpenVINO-model-inference-with-Optimum-intel)


<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/qwen3-vl-embedding/qwen3-vl-embedding.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and is using a custom branch of optimum-intel. It may be fully supported and validated in the future.

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [1]:
import platform

%pip uninstall -q -y optimum optimum-intel optimum-onnx
%pip install -q "git+https://github.com/openvino-dev-samples/optimum-intel.git@qwen3vl-reranker" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "transformers==5.8" "torch>=2.9" "torchvision" "qwen-vl-utils>=0.0.14" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -qU "openvino>=2025.4" "openvino_tokenizers>=2025.4"

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0.0"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
optimum-intel 1.27.0.dev0+8d4d58c requires transformers<5.1,>=4.45, but you have transformers 5.8.0 which is incompatible.
optimum-onnx 0.1.0.dev0 requires transformers<5.6,>=4.36, but you have transformers 5.8.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import requests
from pathlib import Path

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3-vl-embedding.ipynb")

## Select model
[back to top ⬆️](#Table-of-contents:)

Qwen3-VL Embedding Model list:

| Model Type | Models | Size | Layers | Sequence Length | Embedding Dimension | MRL Support | Instruction Aware |
|---|---|---|---|---|---|---|---|
| Multimodal Embedding | [Qwen3-VL-Embedding-2B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B) | 2B | 28 | 32K | 2048 | Yes | Yes |
| Multimodal Embedding | [Qwen3-VL-Embedding-8B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-8B) | 8B | 36 | 32K | 4096 | Yes | Yes |

In [5]:
import ipywidgets as widgets

model_ids = ["Qwen/Qwen3-VL-Embedding-2B", "Qwen/Qwen3-VL-Embedding-8B"]

model_selector = widgets.Dropdown(
    options=model_ids,
    default=model_ids[0],
    description="Embedding Model:",
)

model_selector

Dropdown(description='Embedding Model:', options=('Qwen/Qwen3-VL-Embedding-2B', 'Qwen/Qwen3-VL-Embedding-8B'),…

## Convert model using Optimum Intel
[back to top ⬆️](#Table-of-contents:)

For convenience, we will use OpenVINO integration with HuggingFace Optimum. [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the Transformers and Diffusers libraries and the different tools and libraries provided by Intel to accelerate end-to-end pipelines on Intel architectures.

Among other use cases, Optimum Intel provides a simple interface to optimize your Transformers and Diffusers models, convert them to the OpenVINO Intermediate Representation (IR) format and run inference using OpenVINO Runtime. `optimum-cli` provides command line interface for model conversion and optimization.

General command format:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <output_dir>
```

where task is task to export the model for. Additionally, you can specify weights compression using `--weight-format` argument with one of following options: `fp32`, `fp16`, `int8` and `int4`.

In [6]:
to_compress = widgets.Checkbox(
    value=False,
    description="Weight compression",
    disabled=False,
)

visible_widgets = [to_compress]

options = widgets.VBox(visible_widgets)

options

The Qwen3-VL-Embedding model can be exported by `feature-extraction` task with Optimum-intel.

In [7]:
from pathlib import Path

model_id = model_selector.value

model_base_dir = Path(model_id.split("/")[-1])
additional_args = {"task": "feature-extraction"}

if to_compress.value:
    model_dir = model_base_dir / "INT8"
    additional_args.update({"weight-format": "int8"})
else:
    model_dir = model_base_dir / "FP16"
    additional_args.update({"weight-format": "fp16"})

In [8]:
from cmd_helper import optimum_cli

if not model_dir.exists():
    optimum_cli(model_id, model_dir, additional_args=additional_args)

## Run OpenVINO model inference with Optimum-intel
[back to top ⬆️](#Table-of-contents:)

Select device from dropdown list for running inference using OpenVINO.

In [9]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

The Qwen3-VL-Embedding model can be loaded by class `OVModelForFeatureExtraction` with Optimum-intel.

In [10]:
from optimum.intel import OVModelForFeatureExtraction

model = OVModelForFeatureExtraction.from_pretrained(model_dir, device=device.value, export=False)

In [11]:
import torch
import torch.nn.functional as F

from torch import Tensor
from transformers import AutoProcessor
from qwen_vl_utils import process_vision_info


def last_token_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


def get_embedding(model, processor, inp, instruction="Represent the user's input."):
    """Get normalized embedding for a single input (text, image, or text+image)."""
    content = []
    if isinstance(inp, dict):
        if "image" in inp:
            content.append({"type": "image", "image": inp["image"]})
        if "text" in inp:
            content.append({"type": "text", "text": inp["text"]})
    elif isinstance(inp, str):
        content.append({"type": "text", "text": inp})

    conversation = [
        {"role": "system", "content": [{"type": "text", "text": instruction}]},
        {"role": "user", "content": content},
    ]
    prompt = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    image_inputs = process_vision_info(conversation)[0]
    if image_inputs:
        inputs = processor(text=[prompt], images=image_inputs, return_tensors="pt")
    else:
        inputs = processor(text=[prompt], return_tensors="pt")

    outputs = model(**inputs)
    hidden_state = outputs.last_hidden_state
    if not isinstance(hidden_state, torch.Tensor):
        hidden_state = torch.tensor(hidden_state)
    embedding = last_token_pool(hidden_state, inputs["attention_mask"])
    return F.normalize(embedding, p=2, dim=1)


processor = AutoProcessor.from_pretrained(model_id)

# Example from https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B#basic-usage-example
queries = [
    {"text": "A woman playing with her dog on a beach at sunset."},
    {"text": "Pet owner training dog outdoors near water."},
    {"text": "Woman surfing on waves during a sunny day."},
    {"text": "City skyline view from a high-rise building at night."},
]

documents = [
    {
        "text": "A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset, as the dog offers its paw in a heartwarming display of companionship and trust."
    },
    {"image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"},
    {
        "text": "A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset, as the dog offers its paw in a heartwarming display of companionship and trust.",
        "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
    },
]

# Inputs mix text, image, and text+image modalities so we compute embeddings one at a time.
all_inputs = queries + documents
embeddings = []
for inp in all_inputs:
    emb = get_embedding(model, processor, inp)
    embeddings.append(emb)

embeddings = torch.cat(embeddings, dim=0)

num_queries = len(queries)
query_embeddings = embeddings[:num_queries]
doc_embeddings = embeddings[num_queries:]

# Compute similarity scores
scores = query_embeddings @ doc_embeddings.T
print("Similarity scores (queries x documents):")
print(scores.tolist())

Similarity scores (queries x documents):
[[0.8158053159713745, 0.7108447551727295, 0.7002450227737427], [0.5194921493530273, 0.3300579786300659, 0.44439685344696045], [0.38861438632011414, 0.29249441623687744, 0.32736167311668396], [0.10949830710887909, 0.04480063542723656, 0.08437502384185791]]


### Compare with original PyTorch model (optional)
[back to top ⬆️](#Table-of-contents:)

Tick the checkbox below to also load the original PyTorch FP32 model and print the two score matrices side-by-side. OpenVINO FP16 should agree with PyTorch FP32 within FP16 tolerance (differences of a few thousandths on this example).

> **Note:** this cell is **disabled by default** because loading the full FP32 model requires ~8 GB of memory and adds a few minutes of runtime.


In [ ]:
import ipywidgets as widgets

run_pt_compare_widget = widgets.Checkbox(
    value=False,
    description="Run PyTorch FP32 comparison",
    disabled=False,
)

run_pt_compare_widget

In [12]:
if not run_pt_compare_widget.value:
    print("Skipping PyTorch comparison. Tick the checkbox above and re-run this cell to enable.")
else:
    import inspect

    from transformers import AutoModel

    def get_embedding_pt(model, processor, inp, instruction="Represent the user's input."):
        """PyTorch variant: strip processor keys the Qwen3VLModel forward does not accept
        (e.g. ``mm_token_type_ids`` on transformers < 5.8). Mirrors :func:`get_embedding`."""
        content = []
        if isinstance(inp, dict):
            if "image" in inp:
                content.append({"type": "image", "image": inp["image"]})
            if "text" in inp:
                content.append({"type": "text", "text": inp["text"]})
        elif isinstance(inp, str):
            content.append({"type": "text", "text": inp})

        conversation = [
            {"role": "system", "content": [{"type": "text", "text": instruction}]},
            {"role": "user", "content": content},
        ]
        prompt = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
        image_inputs = process_vision_info(conversation)[0]
        if image_inputs:
            inputs = processor(text=[prompt], images=image_inputs, return_tensors="pt")
        else:
            inputs = processor(text=[prompt], return_tensors="pt")

        accepted = set(inspect.signature(model.forward).parameters)
        call_kwargs = {k: v for k, v in inputs.items() if k in accepted}

        with torch.no_grad():
            outputs = model(**call_kwargs)
        hidden_state = outputs.last_hidden_state
        embedding = last_token_pool(hidden_state, inputs["attention_mask"])
        return F.normalize(embedding, p=2, dim=1)

    # AutoModel loads Qwen3VLModel (the backbone without LM head), matching what
    # OVModelForFeatureExtraction wraps on the OV side.
    pt_model = AutoModel.from_pretrained(model_id, torch_dtype=torch.float32).eval()

    pt_embeddings = []
    for inp in all_inputs:
        pt_embeddings.append(get_embedding_pt(pt_model, processor, inp))
    pt_embeddings = torch.cat(pt_embeddings, dim=0)
    pt_scores = pt_embeddings[:num_queries] @ pt_embeddings[num_queries:].T

    print("PyTorch FP32 similarity scores:")
    print([[round(v, 4) for v in row] for row in pt_scores.tolist()])
    print("OpenVINO FP16 similarity scores:")
    print([[round(v, 4) for v in row] for row in scores.tolist()])

    max_abs_diff = (pt_scores - scores).abs().max().item()
    print(f"Max |PT - OV| across the score matrix: {max_abs_diff:.4f}")

    del pt_model
    import gc

    gc.collect()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

PyTorch FP32 similarity scores:
[[0.8158, 0.712, 0.7015], [0.5195, 0.3296, 0.4442], [0.3884, 0.2934, 0.3283], [0.1093, 0.0449, 0.0843]]
OpenVINO FP16 similarity scores:
[[0.8158, 0.7108, 0.7002], [0.5195, 0.3301, 0.4444], [0.3886, 0.2925, 0.3274], [0.1095, 0.0448, 0.0844]]
Max |PT - OV| across the score matrix: 0.0013


0